# Notebook for generating onboarding related csvs for BI Reports

In [ ]:
%pip install pandas
%pip install sqlalchemy>=2.0
%pip install dotenv
%pip install psycopg2-binary

## Import Libraries and Load Configuration

Import required libraries and load environment variables.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError, OperationalError
from dotenv import load_dotenv

load_dotenv()

print("Libraries imported and configuration loaded successfully.")

## Database Setup

Configure database connections using environment variables.

In [ ]:
DATABASE_CONFIG = {
    'colin_extract': {
        'username': os.getenv("DATABASE_COLIN_EXTRACT_USERNAME"),
        'password': os.getenv("DATABASE_COLIN_EXTRACT_PASSWORD"),
        'host': os.getenv("DATABASE_COLIN_EXTRACT_HOST"),
        'port': os.getenv("DATABASE_COLIN_EXTRACT_PORT"),
        'name': os.getenv("DATABASE_COLIN_EXTRACT_NAME")
    }
}


for db_key, db_config in DATABASE_CONFIG.items():
    uri = f"postgresql://{db_config['username']}:{db_config['password']}@{db_config['host']}:{db_config['port']}/{db_config['name']}"
    DATABASE_CONFIG[db_key] = {'uri': uri}

print("Database configurations successfully.")


## Create Database Engines

Create and test database connections for all configured databases.

In [ ]:
engines = {}

for db_key, config in DATABASE_CONFIG.items():
    try:
        engine = create_engine(config['uri'])

        # Test connection
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))

        engines[db_key] = engine
        print(f"{db_key.upper()} database engine created and tested successfully.")

    except OperationalError as e:
        print(f"{db_key.upper()} database connection failed: {e}")
        raise
    except SQLAlchemyError as e:
        print(f"{db_key.upper()} database engine creation failed: {e}")
        raise
    except Exception as e:
        print(f"{db_key.upper()} unexpected error: {e}")
        raise

ENGINE_NAMES = {engine: key for key, engine in engines.items()}

print("All database engines ready for use.")


## Query active/historical legacy data

Comment out `is_active` and `not is_active` as required to generate the df required

In [ ]:
legacy_corps_query = f"""
select
       mig_group_display_name as mig_group_name,
       mig_batch_display_name as mig_batch_name,
       migrated,
       mig_date,
       email_domain,
       admin_email,
       email_used_count,
       corp_num,
       corp_name,
       corp_type_cd,
       is_active,
       in_dissolution,
       meets_main_criteria,
       has_officers,
       has_3rd_party,
       recognition_dts,
       last_ar_filed_dt,
       director_count,
       filing_cnt,
       months_since_last_ar_filing,
       ar_unfiled_over_1yr,
       file_cnt_last_2yrs,
       last_file_event_ts,
       last_event_ts,
       event_file_types,
       failed_events,
       vendor,
       has_password,
       send_ar_ind,
       meets_share_criteria,
       has_other_share_currency,
       has_share_par_value_lt1_gt6dp,
       has_share_par_value_gt1_gt2dp,
       address_all_any_bad_count,
       office_all_any_bad_count,
       party_all_any_bad_count,
       is_frozen,
       has_bar_filing,
       last_bar_fiscal_year,
       last_bar_filing_date,
       months_since_last_bar_filing,
       bar_account_id,
       bar_account_has_mailing_address,
       group_name
from mv_legacy_corps_data
where 1 = 1
and is_active
-- and not is_active
"""

try:
    with engines['colin_extract'].connect() as conn:
        legacy_corps_df = pd.read_sql(legacy_corps_query, conn)

    if legacy_corps_df.empty:
        raise ValueError("COLIN Extract database query returned empty result")

    print(f"Fetched {len(legacy_corps_df)} rows from COLIN Extract database.")

except Exception as e:
    print(f"Error fetching data from COLIN Extract: {e}")
    raise

# Display results
with pd.option_context('display.max_rows', 5):
    display(legacy_corps_query)

## Generate CSV for BI report

Comment/uncomment csv to generate as required.

In [ ]:

# legacy_corps_df.to_csv('../generated/legacy_corps_data.csv', index=False, na_rep="")
legacy_corps_df.to_csv('../generated/legacy_corps_data_active.csv', index=False, na_rep="")
# legacy_corps_df.to_csv('../generated/legacy_corps_data_historical.csv', index=False, na_rep="")
